# NB05 — The Neurosymbolic Loop

**This notebook demonstrates:**

1. Using an LLM to translate natural-language questions to SPARQL, **validated first by ARQ parsing + predicate whitelist**, then executed against Fuseki
2. Running 3 requests concurrently with `OllamaCloudClient.parallel` to verify the Semaphore(3) ceiling in action
3. Reconstructing a **proof chain** by walking the SWRL rule structure in reverse over the materialized graph (`explain_decision` → structured dict)
4. Feeding the proof chain to `glm-5.1:cloud` to synthesize a human-readable explanation
5. Using `gemini-3-flash-preview:cloud` as a judge to verify the synthesis does not hallucinate
6. Running a **3-question comparison experiment**: Pure LLM vs Neurosymbolic — scored on factuality, determinism, and explainability

## Prerequisites

- Phases A-C verified
- Ollama local daemon running at `http://127.0.0.1:11434` (`brew services start ollama`).
  **Notebook works even if daemon is down** — critical cells automatically fall back to mock httpx; architecture demonstration remains complete.

## Note

Semaphore(3) is a project-level invariant — matches Ollama Cloud Pro quota and avoids saturating the M4 cores under local inference.
To change it, edit `OLLAMA_CONCURRENCY` in `scripts/ollama_client.py`. **Do not** bypass it at the call site.

## 0. Setup & daemon detection

> 🔧 **Tech**: Async LLM client bootstrap + daemon liveness probe + mock httpx transport
> 🎯 **Goal**: Detect local daemon, configure LIVE/mock dual-track, define namespace and canned-response mock factory
> ✅ **Verify**: Prints "reachable — live mode" or "unreachable — mock mode"; FAST/DEEP model names carry `:cloud` suffix
> 📚 **Takeaway**: A neurosymbolic demo must be daemon-agnostic — the symbolic path always runs; the LLM path can be mocked

In [1]:
import os, sys, json, asyncio, tempfile
from pathlib import Path
from dataclasses import dataclass
from rdflib import Graph, Namespace, URIRef, Literal
import httpx
from dotenv import load_dotenv

load_dotenv()

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT / "scripts"))
from ollama_client import OllamaCloudClient, LLMRequest, fast_model, deep_model

def _daemon_reachable() -> bool:
    try:
        r = httpx.get(os.environ.get("OLLAMA_HOST", "http://127.0.0.1:11434").rstrip("/") + "/api/tags",
                      timeout=1.5)
        return r.status_code == 200
    except Exception:
        return False

LIVE = _daemon_reachable()
print(f"Ollama daemon {'reachable — live mode' if LIVE else 'unreachable — mock mode'}")
print(f"FAST model:  {fast_model()}")
print(f"DEEP model:  {deep_model()}")

CR = Namespace("https://nikko.dev/ontology/credit#")
NS_STR = "https://nikko.dev/ontology/credit#"


def make_mock_transport(canned: dict[str, str]) -> httpx.AsyncClient:
    """Return an AsyncClient that returns canned[prompt_substring] match."""
    async def handler(request: httpx.Request) -> httpx.Response:
        await asyncio.sleep(0.05)
        body = json.loads(request.content)
        prompt = body.get("prompt", "")
        for key, ans in canned.items():
            if key in prompt:
                return httpx.Response(200, json={"response": ans})
        return httpx.Response(200, json={"response": "(no canned answer)"})
    return httpx.AsyncClient(transport=httpx.MockTransport(handler))

Ollama daemon reachable — live mode
FAST model:  gemini-3-flash-preview:cloud
DEEP model:  glm-5.1:cloud


## 1. Does Semaphore(3) actually enforce the ceiling?

No matter how many calls a caller fans out with `gather(*[...])`, the client's internal semaphore
should cap active requests at <= 3.
Use mock transport simulating 200 ms API latency, fire 10 parallel requests, observe peak concurrency.

> 🔧 **Tech**: Async concurrency ceiling test — `asyncio.Semaphore(3)` invariant verification
> 🎯 **Goal**: Fire 10 parallel LLM requests; verify the client-internal semaphore caps active count at <= 3
> ✅ **Verify**: `peak == 3` assert passes; all 10 requests return
> 📚 **Takeaway**: Project-level invariants belong in the client layer — a caller can fan out arbitrarily and still not break the ceiling

In [2]:
async def test_concurrency_ceiling():
    async def slow_handler(request):
        await asyncio.sleep(0.2)
        return httpx.Response(200, json={"response": "ok"})
    async with httpx.AsyncClient(transport=httpx.MockTransport(slow_handler)) as ac:
        cli = OllamaCloudClient(_transport_client=ac)
        async with cli:
            reqs = [LLMRequest(model="glm-5.1:cloud", prompt=f"q{i}") for i in range(10)]
            outs = await cli.parallel(*reqs)
    return cli._max_in_flight, len(outs)

peak, n = await test_concurrency_ceiling()
print(f"10 requests issued, peak concurrent = {peak}, returned = {n}")
assert peak == 3, f"semaphore broken: saw {peak}"
print("Semaphore(3) ceiling holds")

10 requests issued, peak concurrent = 3, returned = 10
Semaphore(3) ceiling holds


## 2. NL → SPARQL — schema-aware prompting

Give the LLM a system prompt containing the ontology's core vocabulary (class names, property names) and ask it to emit valid SPARQL.
We **do not blindly trust** the LLM output — we gate it locally:

1. **Syntax**: run it through rdflib's SPARQL parser
2. **Vocabulary**: every predicate the LLM referenced must be declared in the T-Box (guards against hallucination)

> 🔧 **Tech**: T-Box predicate whitelist extraction — symbolic ground truth for LLM output validation
> 🎯 **Goal**: Extract all valid predicate IRIs from `credit_risk.ttl`, augment with RDF/RDFS builtins, form the whitelist
> ✅ **Verify**: Prints whitelist size (>= total object/datatype properties declared in the T-Box)
> 📚 **Takeaway**: The symbolic system acts as a gatekeeper — the whitelist is the hard constraint that decides whether LLM output can enter Fuseki

In [3]:
# Extract all valid predicates from the ontology as a whitelist
tbox = Graph(); tbox.parse(str(PROJECT / "ontology" / "credit_risk.ttl"), format="turtle")
from rdflib import OWL, RDFS, RDF
valid_predicates = set()
for p in tbox.subjects(RDF.type, OWL.ObjectProperty):
    valid_predicates.add(str(p))
for p in tbox.subjects(RDF.type, OWL.DatatypeProperty):
    valid_predicates.add(str(p))
# Add common RDF/RDFS/OWL/SHACL props that any SPARQL might use
valid_predicates |= {
    "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
    "http://www.w3.org/2000/01/rdf-schema#label",
    "http://www.w3.org/2000/01/rdf-schema#subClassOf",
}
print(f"Whitelist: {len(valid_predicates)} predicates")

Whitelist: 23 predicates


> 🔧 **Tech**: SPARQL whitelist validator (syntax via `prepareQuery` + IRI regex scan)
> 🎯 **Goal**: Implement `validate_sparql`, smoke-test 3 cases: good / bad-syntax / bad-predicate
> ✅ **Verify**: good → ok=True; bad syntax → ok=False (syntax error); bad predicate → ok=False (`:hallucinatedProperty` rejected)
> 📚 **Takeaway**: The "verify" half of trust-but-verify — LLM is flexible but untrusted; the validator decides what gets executed

In [4]:
# Validator
def validate_sparql(query: str, whitelist: set[str]) -> tuple[bool, str]:
    """Returns (ok, reason). Fails if syntax broken OR any predicate not whitelisted."""
    from rdflib.plugins.sparql import prepareQuery
    try:
        prepareQuery(query)
    except Exception as e:
        return False, f"syntax: {type(e).__name__}: {str(e)[:200]}"
    # Find IRIs that look like predicates (in p position).
    # Crude but effective: strip PREFIX declarations first, then extract usages.
    import re
    prefixes = dict(re.findall(r'PREFIX\s+(\w*):\s*<([^>]+)>', query))
    body = re.sub(r'PREFIX\s+\w*:\s*<[^>]+>', '', query)
    iris = set(re.findall(r'<(https?://[^>]+)>', body))
    iris_resolved = set()
    for tok in re.findall(r'(?:^|\s)(\w*):([A-Za-z_][\w-]*)', body):
        prefix, local = tok
        if prefix in prefixes:
            iris_resolved.add(prefixes[prefix] + local)
    # Subset of IRIs that appear in property position is hard to extract
    # without a real algebra walk; we do a permissive check: any IRI used
    # in the query must be a valid predicate OR a class/individual.
    classes = {str(c) for c in tbox.subjects(RDF.type, OWL.Class)}
    individuals = {str(i) for i in tbox.subjects(RDF.type, OWL.NamedIndividual)}
    allowed = whitelist | classes | individuals | {
        "http://www.w3.org/1999/02/22-rdf-syntax-ns#type",
    }
    unknown = [iri for iri in (iris | iris_resolved) if iri.startswith("https://nikko.dev") and iri not in allowed]
    if unknown:
        return False, f"unknown predicates/classes: {unknown[:3]}"
    return True, "ok"

# Smoke test the validator
good = '''
PREFIX : <https://nikko.dev/ontology/credit#>
SELECT ?a WHERE { ?a a :Applicant ; :hasCreditScore ?s . FILTER(?s > 700) }
'''
bad_syntax = "SELECT WHERE { ?s :hasFoo ?o"  # missing closing brace
bad_predicate = '''
PREFIX : <https://nikko.dev/ontology/credit#>
SELECT ?a WHERE { ?a :hallucinatedProperty ?x }
'''

for label, q in [("good", good), ("bad syntax", bad_syntax), ("bad predicate", bad_predicate)]:
    ok, reason = validate_sparql(q, valid_predicates)
    print(f"{label:15} -> ok={ok}  ({reason})")

good            -> ok=True  (ok)
bad syntax      -> ok=False  (syntax: ParseException: Expected SelectQuery, found 'WHERE'  (at char 7), (line:1, col:8))
bad predicate   -> ok=False  (unknown predicates/classes: ['https://nikko.dev/ontology/credit#hallucinatedProperty'])


**Note**: The predicate whitelist check above is a **coarse approximation** — it only checks whether prefixed names appear in the T-Box.
A strict implementation would parse the SPARQL algebra, isolate IRIs in *p-position*, and validate those.
This notebook skips that step for clarity.

## 3. Live NL → SPARQL call (live or mock)

If the daemon is running, the notebook calls `gemini-3-flash-preview:cloud` (`gemini-3-flash` is a non-thinking model;
responses go directly into the `response` field). If the daemon is down, canned mock responses are used.
The architecture is identical on both paths.

> 🔧 **Tech**: Schema-aware prompting + `strip_fence` + trust-but-verify fallback
> 🎯 **Goal**: Translate "Which applicants have FICO over 750?" to SPARQL, run the validator, fall back to a hand-written query if rejected
> ✅ **Verify**: Prints the generated SPARQL + validator result; if rejected, prints the fallback warning
> 📚 **Takeaway**: LLM output is a candidate; the symbolic validator is the gatekeeper — graceful degradation on rejection

In [5]:
SCHEMA_DESC = '''
Ontology classes:
  :Applicant, :PrimeApplicant, :NearPrimeApplicant, :SubprimeApplicant,
  :CreditApplication, :MortgageApplication, :PersonalLoanApplication, :AutoLoanApplication,
  :LowRiskApplication, :MediumRiskApplication, :HighRiskApplication,
  :CreditScoreSignal, :EmploymentStabilitySignal, :CreditBureau

Object properties:
  :hasApplicant, :hasCoapplicant, :emittedSignal, :employedBy, :reportedBy,
  :hasDecision, :hasRiskTier

Datatype properties:
  :hasCreditScore (xsd:integer 300-850), :hasAnnualIncome, :hasDebtToIncomeRatio,
  :hasEmploymentYears, :requestedAmount, :requestedTermMonths,
  :signalValue (xsd:decimal 0-1), :signalTimestamp

Named individuals:
  :Approve, :Review, :Decline (instances of :Decision)
'''

SYSTEM = f'''You translate questions to SPARQL 1.1 against this ontology.

Required PREFIX (DO NOT invent your own namespace):
PREFIX : <https://nikko.dev/ontology/credit#>

{SCHEMA_DESC}

Return ONLY the SPARQL query starting with the PREFIX line. No explanation. No ```markdown fences```.'''

def strip_fence(text: str) -> str:
    """Strip ```sparql ... ``` markdown fences if the LLM wrapped its output."""
    import re
    t = text.strip()
    # Remove opening ```anything fence
    t = re.sub(r"^```\w*\s*", "", t)
    # Remove closing ``` fence
    t = re.sub(r"\s*```$", "", t)
    return t.strip()


async def ask_for_sparql(question: str) -> str:
    canned = {
        "FICO over 750": '''PREFIX : <https://nikko.dev/ontology/credit#>
SELECT ?a ?fico WHERE { ?a a :Applicant ; :hasCreditScore ?fico . FILTER(?fico > 750) }''',
        "declined": '''PREFIX : <https://nikko.dev/ontology/credit#>
SELECT ?app WHERE { ?app :hasDecision :Decline }''',
    }
    if LIVE:
        async with OllamaCloudClient() as cli:
            raw = await cli.call(LLMRequest(
                model=fast_model(), system=SYSTEM, prompt=question,
                # gemini-3-flash-preview is ALSO a thinking model -- needs budget
                temperature=0.0, num_predict=1500))
    else:
        async with make_mock_transport(canned) as ac:
            cli = OllamaCloudClient(_transport_client=ac)
            async with cli:
                raw = await cli.call(LLMRequest(
                    model=fast_model(), system=SYSTEM, prompt=question,
                    temperature=0.0, num_predict=1500))
    return strip_fence(raw)

q = "Which applicants have FICO over 750?"
sparql = await ask_for_sparql(q)
print("--- Generated SPARQL ---")
print(sparql)
print("--- Validation ---")
ok, reason = validate_sparql(sparql, valid_predicates)
print(f"ok={ok}  {reason}")

# If the LLM produced garbage, fall back to a hand-written equivalent so the
# rest of the notebook can continue. This is the "trust but verify" pattern.
if not ok:
    print("WARNING: LLM output rejected -- using hand-written fallback")
    sparql = '''PREFIX : <https://nikko.dev/ontology/credit#>
SELECT ?a ?fico WHERE { ?a a :Applicant ; :hasCreditScore ?fico . FILTER(?fico > 750) }'''

--- Generated SPARQL ---
PREFIX : <https://nikko.dev/ontology/credit#>
SELECT ?applicant WHERE {
  ?applicant :hasCreditScore ?score .
  FILTER(?score > 750)
}
--- Validation ---
ok=True  ok


## 4. NL → SPARQL → Fuseki — closing the loop

Send the validated SPARQL to Fuseki and return the results to the LLM for natural-language synthesis.

> 🔧 **Tech**: SPARQL HTTP execution — only a validated query reaches the triple store
> 🎯 **Goal**: Execute the validated SPARQL, convert Fuseki JSON bindings to a list of Python dicts
> ✅ **Verify**: Prints row count + first 5 bindings (applicant + FICO)
> 📚 **Takeaway**: Only queries that pass symbolic validation reach Fuseki — this is the "execute" step in the closed loop

In [6]:
def run_sparql(query: str) -> list[dict]:
    r = httpx.post(
        os.environ.get("FUSEKI_URL", "http://localhost:3030") + "/credit-risk/sparql",
        data={"query": query},
        headers={"Accept": "application/sparql-results+json"},
        timeout=15.0,
    )
    r.raise_for_status()
    j = r.json()
    cols = j["head"]["vars"]
    return [{c: b.get(c, {}).get("value") for c in cols} for b in j["results"]["bindings"]]

results = run_sparql(sparql)
print(f"Fuseki returned {len(results)} rows")
for row in results[:5]: print(" ", row)

Fuseki returned 6 rows
  {'applicant': 'https://nikko.dev/ontology/credit#Applicant_P06'}
  {'applicant': 'https://nikko.dev/ontology/credit#Applicant_P07'}
  {'applicant': 'https://nikko.dev/ontology/credit#Applicant_P03'}
  {'applicant': 'https://nikko.dev/ontology/credit#Applicant_P05'}
  {'applicant': 'https://nikko.dev/ontology/credit#Applicant_P02'}


## 5. Proof Chain — reconstructing rule triggers from Pellet inference

SPEC §4.7 design: `:Explanation` individuals are not a direct SWRL output; they are
reconstructed after the fact from the reasoned graph by asking "which rule caused this `:hasDecision`?"
We implement the trace logic manually by reversing the known rules.

> 🔧 **Tech**: Load the Pellet-inferred graph
> 🎯 **Goal**: Read the full triple set produced by `scripts/run_reasoner.py --apply-r5` into memory, as ground truth for proof-chain reconstruction
> ✅ **Verify**: Prints total triple count (should include original + inferred `:hasRiskTier` / `:hasDecision`)
> 📚 **Takeaway**: Symbolic inference output is the data source for proof chains — inference and explanation are decoupled

In [7]:
# Load full inference results (produced by run_reasoner --apply-r5)
inferred_path = PROJECT / "ontology" / "inferred.ttl"
if not inferred_path.exists():
    print(f"WARNING: {inferred_path} not found -- run scripts/run_reasoner.py --apply-r5 first")
else:
    inferred = Graph()
    inferred.parse(str(inferred_path), format="turtle")
    print(f"Inferred graph: {len(inferred)} triples")

Inferred graph: 1255 triples


> 🔧 **Tech**: Proof chain reconstruction — known rules reversed to find trigger conditions in the inferred graph
> 🎯 **Goal**: Implement `explain_decision`; reconstruct rules_fired + evidence for one app per decision type: App_L01 (Approve), App_M09 (Decline), App_A04 (Review)
> ✅ **Verify**: Each app prints a JSON chain; decision/tier/rule chain consistent with SPEC §6 rules
> 📚 **Takeaway**: SPEC §4.7 — `:Explanation` is not a direct SWRL output; it is reconstructed by reverse-walking the inferred graph (avoids rule explosion)

In [8]:
def explain_decision(app_iri: URIRef, g: Graph) -> dict:
    """Reconstruct which rules led to this app's decision.

    Walks the known SWRL/R5 rules in reverse:
      hasDecision=Decline -> R3 -> hasRiskTier=HighRisk -> R2a or R2b
      hasDecision=Approve -> R4 -> hasRiskTier=LowRisk -> R1
      hasDecision=Review  -> R5 (no tier, no decision was assigned)
    """
    chain = {"app": str(app_iri), "rules_fired": [], "evidence": {}}

    decisions = list(g.objects(app_iri, CR.hasDecision))
    if not decisions: return chain
    decision = decisions[0]
    chain["decision"] = str(decision).split("#")[-1]

    tiers = list(g.objects(app_iri, CR.hasRiskTier))
    tier = tiers[0] if tiers else None
    if tier:
        chain["tier"] = str(tier).split("#")[-1]

    if decision == CR.Decline:
        chain["rules_fired"].append("R3 (HighRisk -> Decline)")
        if tier == CR.HighRiskApplication:
            # Was it R2a (Subprime applicant) or R2b (weak signal)?
            applicant = next(g.objects(app_iri, CR.hasApplicant), None)
            if applicant:
                a_types = set(g.objects(applicant, RDF.type))
                if CR.SubprimeApplicant in a_types:
                    chain["rules_fired"].insert(0, "R2a (Subprime applicant -> HighRisk)")
                    chain["evidence"]["applicant"] = str(applicant).split("#")[-1]
                    chain["evidence"]["fico"] = str(next(g.objects(applicant, CR.hasCreditScore), "?"))
            # Or R2b (weak credit signal)
            signals = list(g.objects(app_iri, CR.emittedSignal))
            for s in signals:
                if CR.CreditScoreSignal in g.objects(s, RDF.type):
                    v = next(g.objects(s, CR.signalValue), None)
                    if v is not None and float(v) < 0.3:
                        chain["rules_fired"].insert(0, f"R2b (CreditScoreSignal value={v} < 0.3)")
                        chain["evidence"]["signal_value"] = str(v)
    elif decision == CR.Approve:
        chain["rules_fired"].append("R4 (LowRisk + stable employment + amount <= 100k -> Approve)")
        if tier == CR.LowRiskApplication:
            chain["rules_fired"].insert(0, "R1 (Prime + strong credit signal -> LowRisk)")
    elif decision == CR.Review:
        chain["rules_fired"].append("R5 (default: no other rule fired -> Review)")
    return chain

# Demonstrate proof chains for three different decision types: Approve / Decline / Review
for app_name in ["App_L01", "App_M09", "App_A04"]:
    iri = URIRef(NS_STR + app_name)
    chain = explain_decision(iri, inferred)
    print(f"\n--- {app_name} ---")
    print(json.dumps(chain, indent=2, ensure_ascii=False))


--- App_L01 ---
{
  "app": "https://nikko.dev/ontology/credit#App_L01",
  "rules_fired": [
    "R1 (Prime + strong credit signal -> LowRisk)",
    "R4 (LowRisk + stable employment + amount <= 100k -> Approve)"
  ],
  "evidence": {},
  "decision": "Approve",
  "tier": "LowRiskApplication"
}

--- App_M09 ---
{
  "app": "https://nikko.dev/ontology/credit#App_M09",
  "rules_fired": [
    "R2a (Subprime applicant -> HighRisk)",
    "R3 (HighRisk -> Decline)"
  ],
  "evidence": {
    "applicant": "Applicant_S04",
    "fico": "510"
  },
  "decision": "Decline",
  "tier": "HighRiskApplication"
}

--- App_A04 ---
{
  "app": "https://nikko.dev/ontology/credit#App_A04",
  "rules_fired": [
    "R5 (default: no other rule fired -> Review)"
  ],
  "evidence": {},
  "decision": "Review"
}


## 6. Proof chain → natural-language explanation (glm-5.1:cloud)

Feed the JSON above to the deep model to synthesize an explanation readable by a loan officer.

> 🔧 **Tech**: Deep thinking model for NL synthesis — proof chain JSON -> 2-3 sentence explanation
> 🎯 **Goal**: Translate the App_L01 (Approve) / App_M09 (Decline) chain JSON into a plain-English explanation for a loan officer
> ✅ **Verify**: Output contains the decision (approved/declined) and references specific rule numbers (R1/R2a/R4...)
> 📚 **Takeaway**: The LLM here is only doing "translation" — facts come from the symbolic chain; the LLM introduces no new facts

In [9]:
SYNTH_SYSTEM = '''You are a credit-decision explainer. Given a JSON proof chain (rules_fired and evidence),
write a brief, factual explanation of why this application was approved, declined, or flagged for review.
Use 2-3 sentences. Fact-driven. No disclaimers. No elaboration beyond what the chain contains.'''

async def synthesize(chain: dict) -> str:
    prompt = json.dumps(chain, ensure_ascii=False)
    canned = {
        "App_M09": "Application declined. Reason: applicant S04 is in the subprime tier (FICO 510), triggering R2a high-risk classification; high-risk applications are automatically declined.",
        "App_L01": "Application approved. Reason: applicant P01 is prime-tier (FICO 810) with a strong credit signal (>=0.8), classified as low-risk by R1; stable employment and amount within 100k triggered R4 automatic approval.",
    }
    app_id = chain.get("app", "").split("#")[-1]
    if LIVE:
        async with OllamaCloudClient() as cli:
            return await cli.call(LLMRequest(
                model=deep_model(), system=SYNTH_SYSTEM, prompt=prompt,
                temperature=0.2, num_predict=1500))  # thinking model needs budget
    else:
        async with make_mock_transport({app_id: canned.get(app_id, "(canned default)")}) as ac:
            cli = OllamaCloudClient(_transport_client=ac)
            async with cli:
                return await cli.call(LLMRequest(
                    model=deep_model(), system=SYNTH_SYSTEM, prompt=prompt,
                    temperature=0.2, num_predict=1500))

for app_name in ["App_L01", "App_M09"]:
    iri = URIRef(NS_STR + app_name)
    chain = explain_decision(iri, inferred)
    explanation = await synthesize(chain)
    print(f"\n--- {app_name} -> {chain.get('decision')} ---")
    print(explanation)


--- App_L01 -> Approve ---
The application was approved because the applicant's prime status and strong credit signal classified them as low risk. This low-risk designation, combined with stable employment and a requested amount of 100k or less, satisfied the conditions for approval.



--- App_M09 -> Decline ---
The application was declined because the applicant's FICO score of 510 classifies them as a subprime, high-risk applicant. Per the established rules, high-risk applications are automatically declined.


## 7. LLM-as-judge — hallucination check on synthesis

Use the fast model as a judge to verify that all facts in the synthesis can be traced back to the original proof chain.
Score 0-2 (0=hallucination, 1=partially supported, 2=fully grounded).

> 🔧 **Tech**: LLM-as-judge fact-checking — fast non-thinking model compares synthesis against chain
> 🎯 **Goal**: Ask the fast model whether every factual claim in the synthesis has evidence in the chain JSON
> ✅ **Verify**: Prints `{"score": ..., "reason": ...}` — in mock mode should yield score=2
> 📚 **Takeaway**: Dual LLM roles — deep model generates, fast model audits; separating responsibilities reduces hallucination risk

In [10]:
JUDGE_SYSTEM = '''You are a fact-checker. Given a proof chain JSON and a plain-English explanation,
judge whether every factual claim in the explanation is supported by evidence in the JSON.
Return JSON: {"score": 0|1|2, "reason": "..."}
- 2 = all facts grounded in chain
- 1 = most facts grounded, minor embellishment
- 0 = clear hallucination (fabricated numbers or misquoted rules)'''

async def judge(chain: dict, synthesis: str) -> dict:
    prompt = f"Chain:\n{json.dumps(chain, ensure_ascii=False)}\n\nExplanation:\n{synthesis}"
    canned_judge = '{"score": 2, "reason": "All facts reference chain rules/evidence."}'
    if LIVE:
        async with OllamaCloudClient() as cli:
            raw = await cli.call(LLMRequest(model=fast_model(), system=JUDGE_SYSTEM,
                                             prompt=prompt, temperature=0.0, num_predict=1500))
    else:
        async with make_mock_transport({"Chain": canned_judge}) as ac:
            cli = OllamaCloudClient(_transport_client=ac)
            async with cli:
                raw = await cli.call(LLMRequest(model=fast_model(), system=JUDGE_SYSTEM,
                                                 prompt=prompt, temperature=0.0, num_predict=1500))
    try:
        return json.loads(strip_fence(raw))
    except json.JSONDecodeError:
        return {"score": -1, "reason": f"unparseable judge output: {raw[:120]}"}

# Pair the previous syntheses with judge calls
iri = URIRef(NS_STR + "App_M09")
chain = explain_decision(iri, inferred)
synth = await synthesize(chain)
verdict = await judge(chain, synth)
print(f"Judge verdict: {verdict}")

Judge verdict: {'score': 2, 'reason': 'All facts in the explanation, including the FICO score, the classification as subprime/high risk, and the final decline decision, are directly supported by the evidence and rules fired in the JSON.'}


## 8. 3-question comparison experiment — Pure LLM vs Neurosymbolic

| # | Question | Evaluation dimension |
|---|---|---|
| Q1 | Why was App_M09 declined? | Factuality (cites real rules vs. fabricates) |
| Q2 | If the FICO threshold drops from 620 to 600, how many Subprime applicants get reclassified? | Determinism (produces a number vs. refuses/guesses) |
| Q3 | Does any application's decision violate a SHACL constraint? | Explainability (gives a specific example vs. vague generality) |

Q2 and Q3 use live Fuseki + SHACL to produce verifiable answers.
Q1 reuses the proof chain from sections 5-7.

> 🔧 **Tech**: Q1 — factuality comparison: Pure LLM (no grounding) vs Neurosymbolic (proof chain grounding)
> 🎯 **Goal**: Ask "Why was App_M09 declined?" from both systems and print side-by-side
> ✅ **Verify**: Pure LLM gives generic speculation (low credit / low income); Neurosymbolic cites specific rules (R2a, FICO=510)
> 📚 **Takeaway**: Without symbolic grounding, the LLM can only say "possible reasons", not "actual reason".

In [11]:
# Q1 - already demonstrated in sections 5/6/7. Reuse:
iri = URIRef(NS_STR + "App_M09")
chain = explain_decision(iri, inferred)
neurosymbolic_q1 = await synthesize(chain)
pure_llm_canned = "App_M09 was likely declined due to a low credit score, insufficient income, or excessive debt load."  # no rule references
print("Q1 -- Why was App_M09 declined?")
print(f"  Pure LLM:       {pure_llm_canned}")
print(f"  Neurosymbolic:  {neurosymbolic_q1}")

Q1 -- Why was App_M09 declined?
  Pure LLM:       App_M09 was likely declined due to a low credit score, insufficient income, or excessive debt load.
  Neurosymbolic:  The application was declined because the applicant's FICO score of 510 classified them as a subprime and high-risk applicant under rule R2a. Per rule R3, high-risk applications are automatically declined.


> 🔧 **Tech**: Q2 — determinism comparison: counterfactual counting (SPARQL COUNT aggregate)
> 🎯 **Goal**: Count how many Subprime applicants would be reclassified if the FICO threshold drops to 600; Pure LLM refuses vs Neurosymbolic gives a definite number
> ✅ **Verify**: Prints SPARQL-returned count + Pure LLM "cannot answer"
> 📚 **Takeaway**: Counterfactual questions need a database — without ground truth, the LLM can only fabricate or decline

In [12]:
# Q2: counterfactual count -- Pure LLM cannot compute this; Neurosymbolic uses SPARQL
q2_sparql = '''
PREFIX : <https://nikko.dev/ontology/credit#>
SELECT (COUNT(?a) AS ?n) WHERE {
  ?a a :Applicant ; :hasCreditScore ?fico .
  FILTER (?fico >= 600 && ?fico < 620)
}
'''
n = run_sparql(q2_sparql)[0]["n"]
print(f"Q2 -- Dropping FICO threshold 620->600 would reclassify {n} Subprime applicants (full list enumerable)")
print(f"  Pure LLM: typically cannot answer (no database access)")
print(f"  Neurosymbolic: {n} (from SPARQL count, reproducible)")

Q2 -- Dropping FICO threshold 620->600 would reclassify 2 Subprime applicants (full list enumerable)
  Pure LLM: typically cannot answer (no database access)
  Neurosymbolic: 2 (from SPARQL count, reproducible)


> 🔧 **Tech**: Q3 — explainability comparison: SHACL constraint violation detection (inject + validate)
> 🎯 **Goal**: Deliberately add a second decision to App_M01 to create a violation, run SHACL, report the specific node+shape
> ✅ **Verify**: Prints violation count > 0 + first Constraint Violation text excerpt
> 📚 **Takeaway**: Symbolic constraints give precise localization (node + shape); Pure LLM can only give vague generalities

In [13]:
# Q3: SHACL violation detection
from pyshacl import validate
data = Graph()
for f in ["credit_risk.ttl", "instances/customers.ttl", "instances/applications.ttl"]:
    data.parse(str(PROJECT / "ontology" / f), format="turtle")
data.parse(str(inferred_path), format="turtle")  # also include reasoned decisions
shapes = Graph(); shapes.parse(str(PROJECT / "ontology" / "shapes.ttl"), format="turtle")

# Intentionally inject a violation: give an application a second decision
data.add((CR.App_M01, CR.hasDecision, CR.Decline))   # M01 already has Approve

conforms, _, text = validate(data, shacl_graph=shapes, inference="rdfs", advanced=True)
n_violations = text.count("Constraint Violation")
print(f"Q3 -- SHACL violations detected: {n_violations}")
print(f"  Pure LLM: cannot verify; can only guess")
print(f"  Neurosymbolic: pinpoints specific node + violated shape")
# Show first violation excerpt
if not conforms:
    start = text.find("Constraint Violation")
    print(text[start:start+400])

Q3 -- SHACL violations detected: 1
  Pure LLM: cannot verify; can only guess
  Neurosymbolic: pinpoints specific node + violated shape
Constraint Violation in MaxCountConstraintComponent (http://www.w3.org/ns/shacl#MaxCountConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:maxCount Literal("1", datatype=xsd:integer) ; sh:message Literal("An application cannot have multiple decisions", lang=en) ; sh:path :hasDecision ]
	Focus Node: cred:App_M01
	Result Path: :hasDecision
	Message: An application cannot have multiple


**Scoring template** (fill in after running the full notebook):

| Q | Factuality | Determinism | Explainability |
|---|---|---|---|
| Q1 Pure LLM | 0-2 | 0-2 | 0-2 |
| Q1 Neurosymbolic | 0-2 | 0-2 | 0-2 |
| Q2 Pure LLM | | | |
| Q2 Neurosymbolic | | | |
| Q3 Pure LLM | | | |
| Q3 Neurosymbolic | | | |

Expected result: Neurosymbolic matches or outperforms Pure LLM across all three dimensions, with a **clear advantage** on Q2 and Q3.


## You should now be able to ✓

- [ ] Write a schema-injected system prompt that steers LLM output toward valid SPARQL
- [ ] Combine rdflib `prepareQuery` + predicate whitelist for cheap LLM output gating
- [ ] Manually implement proof chain reconstruction (known rules -> reverse-walk for triggering factors)
- [ ] Use a deep model for NL synthesis and a fast model for hallucination checking
- [ ] Explain to a colleague why Pure LLM necessarily fails on counterfactual and validation questions

## Next steps

NB99 (Phase E) assembles these components into an end-to-end demo: from raw application to NL explanation + SHACL report.